# Bài 1 — Hiểu Kev và chạy thử trên Colab/Kaggle
Đọc từng phần rồi chạy cell bên dưới bằng ▶ hoặc Shift+Enter.

**Mục tiêu:** thấy một đầu vào, điểm dự đoán, hai bước học tiếp và các file model được lưu.

Dữ liệu ở đây hoàn toàn **giả định để học thao tác**. Các service S1/S2/S3 thuộc ví dụ, không bổ sung vào inventory nghiên cứu. Sáu mẫu học và ba mẫu kiểm tra dùng cùng một quy tắc minh họa; điểm của chúng không phải kết quả RCA khoa học.

Notebook không đọc hoặc ghi raw/Parquet của dự án, không gọi Modal. Bản này đã kiểm tra cú pháp/dữ liệu bằng CPU; chưa được chạy với trọng số trên GPU Colab/Kaggle.

Trên Colab: chọn Runtime → Change runtime type → GPU. Trên Kaggle: bật Internet và chọn GPU ở Settings/Session options. Bài học dùng một GPU.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import textwrap

IS_KAGGLE = Path("/kaggle/working").exists()
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else (Path("/content") if IS_COLAB else Path.cwd())
LAB_ROOT = WORK_ROOT / "kev_lab"
LAB_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT = LAB_ROOT / ("lesson-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
RUN_ROOT.mkdir()

KEV_CODE_REV = "2074cc44256200bb37540103fa4572901faa10f3"
MODEL_ID = "jaredpalmer/kev-0.8b@9a45d25e"
REPO = LAB_ROOT / "kev"

# Chỉ dùng GPU số 0. Hai GPU không tự gộp bộ nhớ.
RUN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "TOKENIZERS_PARALLELISM": "false"}

def run_cmd(args, cwd=None):
    import shlex
    print("$", shlex.join([str(x) for x in args]), flush=True)
    subprocess.run([str(x) for x in args], cwd=cwd, env=RUN_ENV, check=True)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")

print("Nền tảng:", "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "local"))
print("Thư mục kết quả:", RUN_ROOT)
print("Model bài học:", MODEL_ID)

: 

## 1. Xem máy cloud được cấp GPU gì
Lệnh dưới chỉ xem thông tin, chưa tải model. Nếu không có GPU, kiểm tra lại runtime. Giới hạn/quota và loại GPU tùy tài khoản; notebook không bảo đảm thời lượng miễn phí.

In [ ]:
if shutil.which("nvidia-smi"):
    run_cmd(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"])
else:
    print("Chưa thấy nvidia-smi. Hãy bật GPU rồi kết nối lại runtime trước phần model.")

## 2. Tải code và cài môi trường riêng
Cell này cần Internet và tải dependencies, có thể mất thời gian. Máy cloud chạy công việc; máy cá nhân chỉ hiển thị notebook.

Code được khóa ở một commit; model được khóa ở một revision. Python 3.13 chạy trong thư mục .venv riêng để khớp yêu cầu Kev. Notebook gọi trực tiếp Python này, không thay kernel notebook.

Không cài bằng pip install kev: tên đó trên PyPI là một dự án khác. Không cài các kernel FLA/fused cho bài tập đầu; đường PyTorch có thể chậm hơn.

In [ ]:
if importlib.util.find_spec("uv") is None:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = [sys.executable, "-m", "uv"]

if not REPO.exists():
    run_cmd(["git", "clone", "https://github.com/jaredpalmer/kev.git", REPO])
    run_cmd(["git", "checkout", "--detach", KEV_CODE_REV], cwd=REPO)
else:
    current = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    if current != KEV_CODE_REV:
        raise RuntimeError("Thư mục kev đang ở revision khác. Chọn LAB_ROOT mới; bài học không ghi đè checkout cũ.")

run_cmd([*UV, "sync", "--locked", "--no-dev", "--python", "3.13"], cwd=REPO)
PYTHON = REPO / ".venv" / "bin" / "python"
run_cmd([PYTHON, "--version"])
run_cmd([PYTHON, "-m", "kev.train", "--help"], cwd=REPO)

## 3. Nhìn một ví dụ có đáp án
state là nội dung cho model xem; questions là câu hỏi; criteria là danh sách lựa chọn; label là đáp án dùng khi học.

Cell tạo ví dụ giả định: một service có lỗi kết nối trực tiếp, service khác có triệu chứng khi gọi nó. Đáp án được đặt theo kịch bản của người viết bài học, không phải nhãn xác minh từ dataset. Đây chỉ là bài tập thao tác.

Các mẫu cố ý ngắn để thử mặc định 384 state tokens. Metadata nằm ngoài đầu vào model. Dữ liệu nghiên cứu thật cần audit và chia theo incident/nhóm nguồn trước bước này.

In [ ]:
def demo_record(number, partition):
    keys = ["S1", "S2", "S3"]
    target = keys[number % len(keys)]
    events = []
    for service in keys:
        if service == target:
            events.append(f"{service}: ERROR database connection refused; internal health check failed; count={10 + number}.")
        else:
            events.append(f"{service}: requests to {target} time out; own health check passes.")
    # Đổi thứ tự trình bày cho bài học, không tạo thêm incident nghiên cứu độc lập.
    if number % 2:
        events.reverse()
    identity = f"tutorial-demo-{partition}-{number:03d}"
    return {
        "state": "Observations from a simulated incident:\n" + "\n".join(events),
        "questions": {
            "investigate": {
                "type": "choice",
                "instructions": "Which service should be investigated first based on these observations?",
                "criteria": {service: None for service in keys},
                "label": target,
            }
        },
        "_meta": {
            "id": identity,
            "group_id": identity,
            "source": "tutorial_demo",
            "variant": "clean",
            "label_source": "constructed_tutorial_scenario_NOT_dataset_ground_truth",
        },
    }

DATA = RUN_ROOT / "demo_data"
DATA.mkdir(exist_ok=True)
train_rows = [demo_record(i, "train") for i in range(6)]
check_rows = [demo_record(i, "check") for i in range(6, 9)]
for name, rows in [("train", train_rows), ("check", check_rows)]:
    (DATA / f"{name}.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows),
        encoding="utf-8",
    )

print("Một mẫu có đáp án:")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2))
print("\nSố mẫu:", len(train_rows), "học;", len(check_rows), "kiểm tra thao tác.")
print("Các mẫu này KHÔNG phải split nghiên cứu độc lập.")

## 4. Kiểm tra GPU, cấu hình checkpoint và độ dài dữ liệu
Chạy script bằng Python của Kev. Nó kiểm tra CUDA, thử phép tính FP32 nhỏ, đọc cấu hình checkpoint và đếm token. Model nền đầy đủ chỉ được nạp ở phần dự đoán tiếp theo.

T4 không có BF16 native. Code train hiện hỗ trợ fp32/bf16, chưa có fp16; bài học chọn FP32 với model 0.8B và input ngắn. Chưa có số đo VRAM/thời gian trên tài khoản của bạn. Nếu CUDA hoặc phép tính thử lỗi, dừng tại đây để đọc lỗi.

In [ ]:
preflight_script = textwrap.dedent(r"""
    from pathlib import Path
    import hashlib
    import importlib.metadata as packages
    import json
    import subprocess
    import sys
    import torch
    from kev.checkpoint import Checkpoint
    from kev.data import load_records, materialize
    from kev.model import fits, load_tokenizer

    root = Path(sys.argv[1])
    model_id = sys.argv[2]
    if not torch.cuda.is_available():
        raise RuntimeError("PyTorch của Kev chưa dùng được GPU. Kiểm tra GPU runtime/driver trước khi tải model.")
    torch.cuda.set_device(0)
    x = torch.ones((32, 32), device="cuda", dtype=torch.float32)
    result = x @ x
    torch.cuda.synchronize()
    assert result[0, 0].item() == 32
    del x, result

    cp = Checkpoint(model_id)
    meta = cp.meta
    if meta.weights != "lora" or meta.weights_dtype != "fp32":
        raise RuntimeError("Bài học này được soạn cho checkpoint LoRA với backbone FP32.")
    args = meta.extra["args"]
    tokenizer = load_tokenizer(meta.base, revision=meta.base_revision)
    partitions = {}
    for part in ("train", "check"):
        rows = load_records(root / "demo_data" / f"{part}.jsonl")
        rejected = [r["_meta"]["id"] for r in rows if not fits(materialize(r), tokenizer)]
        if rejected:
            raise RuntimeError(f"Record vượt giới hạn token: {rejected}")
        partitions[part] = len(rows)

    config = {
        "init_from": cp.path,
        "base": meta.base,
        "base_revision": meta.base_revision,
        "lora": meta.lora,
        "head_dim": meta.head_dim,
        "lora_targets": args["lora_targets"],
        "option_isolation": int(meta.option_isolation),
        "special_embeddings": int(meta.special_embeddings),
        "weights_dtype": meta.weights_dtype,
    }
    def digest(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()
    manifest = {
        "purpose": "tutorial_only_not_RCA_research",
        "kev_code_revision": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
        "model_id": model_id,
        "resolved_checkpoint": cp.path,
        "head_sha256": digest(Path(cp.path) / "head.pt"),
        "adapter_sha256": cp.weights_sha256(),
        "python": sys.version,
        "packages": {n: packages.version(n) for n in ("torch", "transformers", "peft", "numpy")},
        "gpu": torch.cuda.get_device_name(0),
        "compute_capability": list(torch.cuda.get_device_capability(0)),
        "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
        "cuda_build": torch.version.cuda,
        "training_precision": "fp32",
        "partitions": partitions,
        "data_sha256": {p: digest(root / "demo_data" / f"{p}.jsonl") for p in partitions},
        "research_evaluation": False,
    }
    for name, obj in [("checkpoint_config.json", config), ("run_manifest.json", manifest)]:
        (root / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(manifest, ensure_ascii=False, indent=2))
    print("Preflight OK. Bước tiếp theo mới nạp model nền để dự đoán.")
""")
(RUN_ROOT / "01_preflight.py").write_text(preflight_script, encoding="utf-8")
run_cmd([PYTHON, RUN_ROOT / "01_preflight.py", RUN_ROOT, MODEL_ID], cwd=REPO)

## 5. Cho Kev có sẵn dự đoán trước khi học
Đây là inference: nạp model → bỏ đáp án khỏi request → tính điểm → sắp lựa chọn. Chưa cập nhật trọng số.

Script dùng api_request → to_record → encode → forward → to_answers, là đường đọc code nên theo dõi đầu tiên. Điểm dùng temperature=1 cho cả trước/sau; bài học chưa hiệu chuẩn xác suất. Các con số in ra là kết quả thật của phiên chạy, không được điền sẵn.

Mỗi lần dự đoán chạy trong một process riêng. Khi process xong, bộ nhớ GPU được giải phóng trước bước training.

In [ ]:
predict_script = textwrap.dedent(r"""
    from pathlib import Path
    import json
    import sys
    import torch
    from kev.api import SystemOneRequest, to_record, to_answers
    from kev.checkpoint import Checkpoint, LoadOptions
    from kev.data import api_request, load_records

    checkpoint, root, name = sys.argv[1], Path(sys.argv[2]), sys.argv[3]
    opts = LoadOptions(dtype=torch.float32, merge=False, backend="torch",
                       fused=False, cuda_graphs=False, temperature=1.0)
    tokenizer, model = Checkpoint(checkpoint).load("cuda", opts)
    predictions = []
    for example in load_records(root / "demo_data" / "check.jsonl"):
        request = SystemOneRequest.model_validate(api_request(example))
        record, question_meta = to_record(request)
        encoded = model.encode(tokenizer, record, strict=True)
        with torch.inference_mode():
            probabilities = [torch.softmax(z.float(), -1).cpu().tolist() for z in model.forward(encoded)]
        keys = question_meta[0]["keys"]
        ranked = sorted(zip(keys, probabilities[0]), key=lambda item: (-item[1], item[0]))
        output = {
            "example_id": example["_meta"]["id"],
            "answers": to_answers(probabilities, question_meta),
            "ranking": [{"service": service, "score": score} for service, score in ranked],
            "purpose": "tutorial_only_NOT_research_metric",
        }
        predictions.append(output)
        print(json.dumps(output, ensure_ascii=False, indent=2))
    (root / f"{name}.json").write_text(
        json.dumps(predictions, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
""")
(RUN_ROOT / "02_predict.py").write_text(predict_script, encoding="utf-8")
config = json.loads((RUN_ROOT / "checkpoint_config.json").read_text(encoding="utf-8"))
run_cmd([PYTHON, RUN_ROOT / "02_predict.py", config["init_from"], RUN_ROOT, "demo_before"], cwd=REPO)

## 6. Học tiếp tối đa hai bước
Fine-tuning khởi tạo từ Kev có sẵn. Ta học adapter LoRA và phần chấm điểm, giữ cố định phần lớn model nền.

Các cờ chính:
- data: file có đáp án để học.
- init_from: checkpoint Kev ban đầu.
- lr 2e-5: mức điều chỉnh trọng số.
- batch 1: xử lý từng mẫu nhỏ.
- accum 2: gom hai lượt nhỏ trước một lần cập nhật.
- max_steps 2: dừng sau tối đa hai lần cập nhật.
- checkpointing 1: tính lại một số phần trung gian để tiết kiệm bộ nhớ.
- out: thư mục lưu model.

Đây là thử kỹ thuật; không kỳ vọng hai bước đủ học RCA. Không dùng các cờ resume định kỳ: source hiện chỉ hỗ trợ các cờ đó cho full-weight training. Nếu phiên bị ngắt trước khi lưu cuối, run LoRA này có thể phải chạy lại.

In [ ]:
CHECKPOINT = RUN_ROOT / "checkpoint"
if CHECKPOINT.exists():
    raise RuntimeError("Đã có thư mục checkpoint của lần thử này. Chạy từ cell đầu để tạo run mới; không ghi đè.")

cmd = [
    PYTHON, "-m", "kev.train",
    "--data", DATA / "train.jsonl",
    "--init_from", config["init_from"],
    "--base", config["base"],
    "--lora", config["lora"],
    "--head_dim", config["head_dim"],
    "--lora_targets", config["lora_targets"],
    "--option_isolation", config["option_isolation"],
    "--special_embeddings", config["special_embeddings"],
    "--weights_dtype", config["weights_dtype"],
    "--device", "cuda", "--dtype", "fp32",
    "--lr", "2e-5", "--epochs", "1",
    "--batch", "1", "--accum", "2",
    "--checkpointing", "1", "--max_steps", "2",
    "--p_none", "0", "--p_none_distract", "0", "--p_distract", "0", "--p_none_pair", "0",
    "--replay", "0", "--seed", "42", "--out", CHECKPOINT,
]
if config["base_revision"]:
    cmd += ["--base_revision", config["base_revision"]]
write_json(RUN_ROOT / "training_command.json", [str(x) for x in cmd])

# Ghi log và hiển thị ngay. Nếu người dùng ngắt cell, dừng process train con.
import shlex
print("$", shlex.join([str(x) for x in cmd]), flush=True)
with (RUN_ROOT / "train.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        [str(x) for x in cmd], cwd=REPO, env=RUN_ENV,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, cmd)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise

required = ["head.pt", "adapter_config.json", "adapter_model.safetensors", "training_metrics.json"]
missing = [name for name in required if not (CHECKPOINT / name).is_file()]
if missing:
    raise RuntimeError(f"Thiếu file kết quả: {missing}")
print("\nĐã lưu checkpoint:", CHECKPOINT)
metrics = json.loads((CHECKPOINT / "training_metrics.json").read_text(encoding="utf-8"))
print("Số bước cập nhật:", metrics["optimizer_steps"])
print("Bộ nhớ GPU peak do trainer báo (GiB):", round(metrics["peak_device_bytes"] / 2**30, 3))

## 7. Dùng model vừa lưu và xem điểm
Lần gọi này đọc checkpoint mới. Không học thêm. Top-1 ở bài tập có thể đổi hoặc giữ nguyên; không suy ra mô hình đã tốt hơn trên incident thật. Muốn kết luận cần tập đánh giá độc lập, baseline và protocol V3.

In [ ]:
run_cmd([PYTHON, RUN_ROOT / "02_predict.py", CHECKPOINT, RUN_ROOT, "demo_after"], cwd=REPO)
before = json.loads((RUN_ROOT / "demo_before.json").read_text(encoding="utf-8"))
after = json.loads((RUN_ROOT / "demo_after.json").read_text(encoding="utf-8"))
assert [row["example_id"] for row in before] == [row["example_id"] for row in after]
for a, b in zip(before, after):
    print(a["example_id"])
    print("  Trước:", a["ranking"])
    print("  Sau:  ", b["ranking"])

## 8. Tải kết quả về máy
ZIP chứa adapter, head.pt, dữ liệu giả định, cấu hình, log và dự đoán của bài học. Nó không chứa toàn bộ model nền hoặc môi trường .venv. Khi dùng lại, Kev tải đúng model nền được checkpoint ghi nhận.

Colab: cell gọi chức năng download. Kaggle: lấy ZIP ở đường dẫn in ra trong phần Files/Output, hoặc lưu phiên bản chạy và kiểm tra Output. Lưu riêng notebook; lưu notebook không tự giữ các file checkpoint trong runtime.

Tài liệu đọc code: docs/notes/kev_bat_dau_colab_kaggle.md trong repo dự án.

In [ ]:
archive = Path(shutil.make_archive(str(LAB_ROOT / f"{RUN_ROOT.name}-outputs"), "zip", root_dir=RUN_ROOT))
print("Tải file này về:", archive)
print("Dung lượng ZIP (MiB):", round(archive.stat().st_size / 2**20, 2))
if IS_COLAB:
    from google.colab import files
    files.download(str(archive))
else:
    print("Mở Files/Output để tải ZIP. Trên Kaggle, kiểm tra output đã lưu trước khi kết thúc phiên.")

## Sau bài học
Bạn cần chỉ ra được: đầu vào nằm ở đâu, label dùng khi nào, inference khác training thế nào và checkpoint được lưu ở đâu.

Bước nghiên cứu tiếp theo là audit RCAEval → split theo incident/nhóm nguồn → JSONL có mapping bằng chứng → baseline → fine-tune → đánh giá theo incident. Notebook này chưa triển khai các bước đó.

Nguồn:
- https://research.google.com/colaboratory/faq.html
- https://www.kaggle.com/docs/efficient-gpu-usage
- https://github.com/jaredpalmer/kev
- https://huggingface.co/jaredpalmer/kev-0.8b